In [0]:
%sql

CREATE OR REPLACE TABLE job_market.gold.hn_yearly_trends_unified AS

SELECT
    posted_year,
    COUNT(*) AS total_posts,

    COUNT_IF(data_source = 'cotera') AS cotera_posts,
    COUNT_IF(data_source = 'api') AS api_posts,

    COUNT_IF(is_remote = true) AS remote_posts,
    COUNT_IF(is_remote = false) AS non_remote_posts,
    COUNT_IF(is_remote IS NULL) AS unknown_remote_posts,

    ROUND(
        100.0 * COUNT_IF(is_remote = true)
        / NULLIF(COUNT_IF(is_remote IS NOT NULL), 0),
        2
    ) AS remote_percentage_known,

    COUNT_IF(SIZE(skills) > 0) AS posts_with_skills,

    ROUND(
        100.0 * COUNT_IF(SIZE(skills) > 0) / COUNT(*),
        2
    ) AS skills_coverage_percentage

FROM job_market.silver.hn_posts_unified
GROUP BY posted_year;


In [0]:
%sql

CREATE OR REPLACE TABLE job_market.gold.hn_skill_trends_unified AS

WITH expanded AS (
    SELECT
        post_id,
        posted_year,
        data_source,
        LOWER(TRIM(tech.skill)) AS skill

    FROM job_market.silver.hn_posts_unified,
    LATERAL EXPLODE(skills) AS tech(skill)
)

SELECT
    posted_year,
    skill,
    COUNT(DISTINCT post_id) AS total_posts,

    COUNT(DISTINCT CASE
        WHEN data_source = 'cotera'
        THEN post_id
    END) AS cotera_posts,

    COUNT(DISTINCT CASE
        WHEN data_source = 'api'
        THEN post_id
    END) AS api_posts

FROM expanded
WHERE skill IS NOT NULL AND skill <> ''
GROUP BY posted_year, skill;


In [0]:
%sql

SELECT
    (SELECT COUNT(*)
     FROM job_market.silver.hn_posts_unified)
         AS silver_posts,

    (SELECT SUM(total_posts)
     FROM job_market.gold.hn_yearly_trends_unified)
         AS gold_posts;


In [0]:
%sql

SELECT COUNT(*)
     FROM job_market.silver.hn_posts_unified
         AS silver_posts
LIMIT 10;         